This program is designed to do the following:
 - It will ingest two datasets, one to train it and one to test it.
 - The data is first cleaned as to ensure a consistent result, as well as an unbiased one
 - A RandomForestClassifier is deployed with 102 decision trees. This helps it reduce variance and boosts its accuracy.
 - It classifies each potential threat into 1 of 10 different classifications such as Normal, DoS, Fuzzer, Reconnaissance etc.
 - The model will have its accuracy rated as a percentage to display how well it did.
 - Two graphs will be generated, one to show an ROC (Receiver Operating Characteristic) curve, and one to show a Precision-Recall curve.

In [ ]:
import pandas as pd # We use pandas to load, clean and manipulate data.
import numpy as np  # This is used for the maths portions.
import seaborn as sns  # Seaborn is used for a bar graph near the start of the program.
import matplotlib.pyplot as plt # This is the skeleton for all the graphs in the program.
%matplotlib inline

# These are mostly for aesthetics, not needed, but adds a bit more sophistication and flair.
import time 
import sys       
import itertools
import threading

# The capstone muscle of the program, needed for all the requirements of the project
from sklearn.preprocessing import LabelEncoder, label_binarize  # We need these to convert multi-class and text labels like 'Worm' and 'Fuzzer' into Binary.
from sklearn.model_selection import train_test_split # Splits the data into an 80% Training and 20% Testing for the last part of the program.
from sklearn.ensemble import RandomForestClassifier # This is the AI algorithm we are using, literally an armada of 102 decision trees, and I cant even decide what I want for tea.
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, average_precision_score, precision_recall_curve
# sklearn.metrics gives us all we need for the guessing, precision-recall, calculating the data needed for the curves. Without them, the graph might as well be blank.



Before we do any training, we need to clean the dataset. AI models on this level for the most part are dumb as rocks. We will be using a Random Forest as the algorithm and we threw a big dataset at it with blank spaces and hyphens, we will likely get an error. It will also reduce the number of False Positives we might get.

This model is only as smart as the data it eats, it really does mirror the 'You are what you eat' philosophy.

In [ ]:
# Loads the CSV into Pandas
train = pd.read_csv('training_dataset1.csv')

In [ ]:
# Displays the metadata of the DataFrame. It just shows that the raw CSV provided was ingested correctly
train.info()

In [ ]:
# This counts the amount of blank spaces (null values) there are within the data
train.isnull().sum()

In [ ]:
# This caluclates the grand total of all the blank spaces (null values) within the dataset
train.isnull().sum().sum()

In [ ]:
# Adds up how many times each specific attack turns up in the dataset. It turns the raw counts into decimals and then multiplies it by 100 to give the %
class_percentages = train['attack_cat'].value_counts(normalize=True) * 100

print("Class distribution in Training set:")

print(class_percentages.round(2).astype(str) + ' %')

This bar graph simply helps visualise the data we will be using. During testing, simply using the matplotlib library made the graph all wonky.
To accomodate this I simply used seaborn for this graph as it worked better, but to keep things consistant with the rest of the program, this is the only place this is used. It overall looks a lot neater an presentable.

In [ ]:
# This is a simple horizontal bar chart to visualise the data

plt.figure(figsize = (10, 6))

plt.style.use('dark_background')

# This time here is the only place Seaborn is used, in context of this bar graph, it just made it look a bit neater than just using matplotlib.
sns.countplot(y = 'attack_cat', data = train, color = '#00E5FF') 

plt.xlabel('Count', fontsize = 12)
plt.ylabel('Attack Category', fontsize = 12)

plt.title('CLASS DISTRIBUTION OF TRAINING DATASET', color = '#00E5FF')
plt.grid(color = '#333333', linestyle = '-', linewidth = 0.5, axis = 'x') 

plt.tight_layout()
plt.show()

In [ ]:
# The next 3 cells function checks the data types of each column and filters it 
train.dtypes.value_counts()


In [ ]:
# Checks which columns contain text data types
train.select_dtypes(exclude=np.number).columns

In [ ]:
# Checks which columns are numerical
train.select_dtypes(include=np.number).columns

In [ ]:
# This cell tallies the frequency of all unique values to verify strict binary distribution (0 and 1) and no anomalies
train['is_sm_ips_ports'].value_counts()

In [ ]:
# This is to identify in anomalies within the dataset, particulary integers or values greater than 1
train['is_ftp_login'].value_counts()


In [ ]:
# This cell includes an if/else statement to cap any value greater than 1 back down to 1
train['is_ftp_login'] = np.where(train['is_ftp_login']>1, 1, train['is_ftp_login'])

In [ ]:
# After the previous cell, this one just confirms that the column has been successfully normalised into a binary format
train['is_ftp_login'].value_counts()

In [ ]:
# Exposes the hyphen used as a placeholder for missing values
train['service'].value_counts()

In [ ]:
# Applies a lambda function to replace the hyphen with 'None'
train['service'] = train['service'].apply(lambda x: "None" if x=="-" else x)

In [ ]:
# Similar to the previous set of cells, this line is ran again to verify that the placerholer has been removed
train['service'].value_counts()

In [ ]:
# The cleaned data is now saved to file
train.to_csv('cleaned_train.csv', index=False)

By this point the folder will now have a saved CSV file. After this point the program will read that file everytime, so you should not need to clean it again, unless you delete the new CSV.

In [ ]:
# Since we have now saved the csv file, we can reload it into the model
train = pd.read_csv('cleaned_train.csv')

# Isolate the Target Vector (y): This is the classifications the model must learn to predict later on
y = train['attack_cat']

# Isolate the Feature Matric (X): Drop the target column so the model only processes the predictive variables
X = train.drop('attack_cat', axis=1)

# This outputs the structural dimensions and verifies to us they are isolated.
print("Shape of X (The Clues):", X.shape)
print("Shape of y (The Answer Key):", y.shape)

In [ ]:
# First we initialise the Label Encoder for the target variable, then turn the text-based attack categories into integer arrays
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Verify target encoding mapping
print("Original text classes:", le.classes_)
print("New numerical classes:", np.unique(y_encoded))


In [ ]:
# drop_first=True ensures linear indpendence among the created dummy variables
X_encoded = pd.get_dummies(X, columns=['proto', 'service', 'state'], drop_first=True)

# Verifies matrix expansion
print("Old shape of X:", X.shape)
print("New shape of X_encoded:", X_encoded.shape)

In [ ]:
# The data is partitioned: 80% Training and 20% Testing
# random_state = 42 (This is an inside joke relating to The Ultimate Hitchhikers guide to the Galaxy, but the number really does not matter in relation to the program)
# I learnt this from LinkedIn from a friend and that its quite common to use in the Data Science industry
# What its meant to do, is lock the algorithmic seed for exact reproducibility every time, across multiple executions
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42)

# Outputs the final structural dimensions before initiating the model training
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

I have decided to seperate the training and testing into two seperate cells for more control when running the program. It makes it easier during testng.
I also added a few extra novelties in here to make it less boring. Didn't need to, but I done it anyway, because its fun.

In [ ]:
print(">>> UPLINK ESTABLISHED. CONNECTION STATUS: ACTIVE.")
time.sleep(1.0)

print(">>> INITIALISING CONSTRUCT: HEIMDALL v1.0") # Decided to call it Heimdall, named after the Norse god of Foresight, I love anything about mythology
time.sleep(1.0)

print(">>> INGESTING NETWORK TRAFFIC...")
time.sleep(1.0)


def spin_animation():
    spinner = itertools.cycle(["-", '\\', '|', '/'])
    while not training_complete:
        sys.stdout.write(f'\r>>> TRAINING... Hold on to your butts![{next(spinner)}]') # A film reference from Jurassic Park, credit goes to the writer David Koepp. Wow, I really am a nerd
        sys.stdout.flush()
        time.sleep(0.1)

training_complete = False
spinner_thread = threading.Thread(target = spin_animation)
spinner_thread.start()



# 102 Trees to make with the random state of 42    
rf_model = RandomForestClassifier(n_estimators=102, random_state=42)                                                                    
rf_model.fit(X_train, y_train)

training_complete = True
spinner_thread.join()

# Prediction
predictions = rf_model.predict(X_test)

# Will give an accuracy score based on the training set
accuracy = accuracy_score(y_test, predictions)
print(f"\n>>>SYSTEM ACCURACY - {accuracy * 100:.2f}%") 

# Once training is complete this message will print out
print(">>> TRAINING COMPLETE.")


The next code block is the main testing cell, where the model is presented with the fresh, testing dataset

In [ ]:
# This is the final code block to test the Model against a fresh set of data
test_data = pd.read_csv('testing_dataset1.csv')

# Similar to the cleaning methods before, we will do the same again on the test dataset
test_data['is_ftp_login'] = np.where(test_data['is_ftp_login'] > 1, 1, test_data['is_ftp_login'])
test_data['service'] = test_data['service'].apply(lambda x: "None" if x=="-" else x)

# Isolate the holdout Feature Matrix and Target Vector
y_holdout = test_data['attack_cat']
X_holdout = test_data.drop('attack_cat', axis=1)

y_holdout_encoded = le.transform(y_holdout)

X_holdout_encoded = pd.get_dummies(X_holdout, columns=['proto', 'service', 'state'], drop_first=True)

# This line forces the test matrix to math the training matrix
# Any columns that are missing in the testing set are filled with zeros
X_holdout_encoded = X_holdout_encoded.reindex(columns=X_train.columns, fill_value=0)

# These lines deploy the model against the alinged holdout matrix
holdout_predictions = rf_model.predict(X_holdout_encoded)

holdout_probabilities = rf_model.predict_proba(X_holdout_encoded)

print(f">>>FINAL HOLDOUT ACCURACY: {accuracy_score(y_holdout_encoded, holdout_predictions) * 100:.2f}%\n")
print(">>>HOLDOUT CLASSIFICATION REPORT:\n")
print(classification_report(y_holdout_encoded, holdout_predictions, target_names=le.classes_))


The following cells are the Reciever Operating Characteristic (ROC) and Precision-Recall (PR) graphs. Both made with the matplotlib library.

In [ ]:
# To get the maths right for the ROC graph, we convert each class into a binary matrix to calculate one specific attack vs all other traffic.

n_classes = len(le.classes_)
y_holdout_bin = label_binarize(y_holdout_encoded, classes = range(n_classes))

plt.style.use('dark_background') # Dark mode, much better to see the figures, at least for myself but I cant speak for everyone.

colors = [         # Each colour corresponds to a specific class of threat.
    '#00E5FF',
    '#FF003C',
    '#00FF41',
    '#FCE205',
    '#BC13FE',
    '#FF6C11',
    '#0B24FB',
    '#FF00FF',
    '#39FF14',
    '#FFFFFF',
]

plt.figure(figsize = (10, 8))

# Iterates through all attack classes to calculate and draw their ROC curves
for i in range(n_classes):
    # Calculates False Positive Rate (x axis) and True Positive Rate (y axis)
    fpr, tpr, _ = roc_curve(y_holdout_bin[:, i], holdout_probabilities[:, i])

    # This calculates the Area Under the Curve for the legend in the bottom right corner
    roc_auc = auc(fpr, tpr)

    # Now we render the curve onto the graph
    plt.plot(fpr, tpr, color = colors[i], lw = 2.5,
             label = f'{le.classes_[i]} (AUC = {roc_auc:.2f})')

# Graph Formatting   
plt.grid(color = "#333333EB", linestyle = '-', linewidth = 0.5)    
plt.plot([0, 1], [0, 1], color = '#555555', linestyle = '--', lw = 2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize = 12)
plt.ylabel('True Positive Rate', fontsize = 12)
plt.title('ROC ANALYSIS PER ATTACK CATEGORY ', fontsize = 14, color = "#FF0000", fontweight = 'bold')
plt.legend(loc = "lower right", fontsize = 10, facecolor = '#111111', edgecolor = '#00E5FF')

plt.tight_layout()
plt.show()


In [ ]:
# Exactly like the ROC curve, the Precision-Recall curve needs binary inputs.
n_classes = len(le.classes_)
y_holdout_bin = label_binarize(y_holdout_encoded, classes = range(n_classes))

plt.style.use('dark_background')

colors = [
    '#00E5FF',
    '#FF003C',
    '#00FF41',
    '#FCE205',
    '#BC13FE',
    '#FF6C11',
    '#0B24FB',
    '#FF00FF',
    '#39FF14',
    '#FFFFFF',
]

plt.figure(figsize = (10, 8))

# Iterate through all the different classes to calculate their specific Precision-Recall curves
for i in range(n_classes):
    # Calculate Precision (y axis) and Recall (x axis) across various algorithmic thresholds
    precision, recall, _ = precision_recall_curve(y_holdout_bin[:, i], holdout_probabilities[:, i])

    # Calculates the Average Precision score 
    avg_prec = average_precision_score(y_holdout_bin[:, i], holdout_probabilities[:, i])

    # Renders the curves onto the graph
    plt.plot(recall, precision, color = colors[i], lw = 2,
             label = f'{le.classes_[i]} (AP = {avg_prec:.2f})')
    
# Graph Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize = 12)
plt.ylabel('Precision', fontsize = 12)
plt.title('PRECISION-RECALL PER ATTACK CATEGORY', color = "#FF0000", fontsize = 14)
plt.legend(loc = "lower left", fontsize = 10, facecolor = '#111111', edgecolor = '#00E5FF')


plt.tight_layout()
plt.show()
